In [0]:
USE CATALOG retail_sales_catalog;
USE SCHEMA gold;


-- REPORT 1: Monthly Sales Revenue
CREATE OR REPLACE VIEW gold.monthly_sales_revenue AS
SELECT
    DATE_FORMAT(f.TxnDate, 'yyyy-MM')        AS Month,
    COUNT(f.TransactionID)                    AS Total_Transactions,
    SUM(f.Quantity)                           AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2)                   AS Total_Revenue
FROM retail_sales_catalog.silver.FactSales f
GROUP BY DATE_FORMAT(f.TxnDate, 'yyyy-MM')
ORDER BY Month;

SELECT * FROM gold.monthly_sales_revenue;


-- REPORT 2: Revenue by Product Category
CREATE OR REPLACE VIEW gold.revenue_by_category AS
SELECT
    p.Category,
    COUNT(f.TransactionID)                    AS Total_Transactions,
    SUM(f.Quantity)                           AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2)                   AS Total_Revenue,
    ROUND(AVG(f.Amount), 2)                   AS Avg_Order_Value
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimProduct p ON f.ProductSK = p.ProductSK
GROUP BY p.Category
ORDER BY Total_Revenue DESC;

SELECT * FROM gold.revenue_by_category;


-- REPORT 3: Top 10 Best Selling Products
CREATE OR REPLACE VIEW gold.top10_products AS
SELECT
    p.ProductID,
    p.ProductName,
    p.Category,
    p.UnitPrice,
    SUM(f.Quantity)                           AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2)                   AS Total_Revenue
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimProduct p ON f.ProductSK = p.ProductSK
GROUP BY p.ProductID, p.ProductName, p.Category, p.UnitPrice
ORDER BY Total_Revenue DESC
LIMIT 10;

SELECT * FROM gold.top10_products;


-- REPORT 4: Store-wise Sales Performance
CREATE OR REPLACE VIEW gold.store_sales_performance AS
SELECT
    st.StoreID,
    st.StoreName,
    st.Region,
    COUNT(f.TransactionID)                    AS Total_Transactions,
    SUM(f.Quantity)                           AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2)                   AS Total_Revenue,
    ROUND(AVG(f.Amount), 2)                   AS Avg_Transaction_Value
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimStore st ON f.StoreSK = st.StoreSK
GROUP BY st.StoreID, st.StoreName, st.Region
ORDER BY Total_Revenue DESC;

SELECT * FROM gold.store_sales_performance;


-- REPORT 5: Region-wise Revenue Summary
CREATE OR REPLACE VIEW gold.region_revenue_summary AS
SELECT
    st.Region,
    COUNT(DISTINCT st.StoreID)                AS Total_Stores,
    COUNT(f.TransactionID)                    AS Total_Transactions,
    SUM(f.Quantity)                           AS Total_Units_Sold,
    ROUND(SUM(f.Amount), 2)                   AS Total_Revenue
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimStore st ON f.StoreSK = st.StoreSK
GROUP BY st.Region
ORDER BY Total_Revenue DESC;

SELECT * FROM gold.region_revenue_summary;


-- REPORT 6: Top 10 Customers by Revenue
CREATE OR REPLACE VIEW gold.top10_customers AS
SELECT
    c.CustomerID,
    c.CustomerName,
    c.City,
    COUNT(f.TransactionID)                    AS Total_Orders,
    SUM(f.Quantity)                           AS Total_Units_Bought,
    ROUND(SUM(f.Amount), 2)                   AS Total_Spent
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimCustomer c
    ON f.CustomerSK = c.CustomerSK
    AND c.IsActive  = 1
GROUP BY c.CustomerID, c.CustomerName, c.City
ORDER BY Total_Spent DESC
LIMIT 10;

SELECT * FROM gold.top10_customers;


-- REPORT 7: Monthly Revenue by Region
CREATE OR REPLACE VIEW gold.monthly_revenue_by_region AS
SELECT
    DATE_FORMAT(f.TxnDate, 'yyyy-MM')         AS Month,
    st.Region,
    COUNT(f.TransactionID)                    AS Total_Transactions,
    ROUND(SUM(f.Amount), 2)                   AS Total_Revenue
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimStore st ON f.StoreSK = st.StoreSK
GROUP BY DATE_FORMAT(f.TxnDate, 'yyyy-MM'), st.Region
ORDER BY Month, Total_Revenue DESC;

SELECT * FROM gold.monthly_revenue_by_region;


-- REPORT 8: SCD2 Customer Change History
CREATE OR REPLACE VIEW gold.customer_change_history AS
SELECT
    CustomerID,
    CustomerName,
    City,
    Address,
    StartDate,
    EndDate,
    CASE IsActive WHEN 1 THEN 'Active' ELSE 'Expired' END AS Status
FROM retail_sales_catalog.silver.DimCustomer
WHERE CustomerID IN (
    SELECT CustomerID
    FROM retail_sales_catalog.silver.DimCustomer
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
)
ORDER BY CustomerID, StartDate;

SELECT * FROM gold.customer_change_history;


-- ADDITIONAL SQL QUERIES (no views — direct business questions)

-- Q1: Which month had the highest revenue?
SELECT
    DATE_FORMAT(TxnDate, 'yyyy-MM')           AS Month,
    ROUND(SUM(Amount), 2)                     AS Total_Revenue
FROM retail_sales_catalog.silver.FactSales
GROUP BY DATE_FORMAT(TxnDate, 'yyyy-MM')
ORDER BY Total_Revenue DESC
LIMIT 1;


-- Q2: Which city has the most active customers?
SELECT
    City,
    COUNT(CustomerID)                         AS Customer_Count
FROM retail_sales_catalog.silver.DimCustomer
WHERE IsActive = 1
GROUP BY City
ORDER BY Customer_Count DESC
LIMIT 5;


-- Q3: Average revenue per transaction by store region
SELECT
    st.Region,
    ROUND(AVG(f.Amount), 2)                   AS Avg_Revenue_Per_Transaction
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimStore st ON f.StoreSK = st.StoreSK
GROUP BY st.Region
ORDER BY Avg_Revenue_Per_Transaction DESC;


-- Q4: Products that have never been sold
SELECT
    p.ProductID,
    p.ProductName,
    p.Category,
    p.UnitPrice
FROM retail_sales_catalog.silver.DimProduct p
LEFT JOIN retail_sales_catalog.silver.FactSales f ON p.ProductSK = f.ProductSK
WHERE f.ProductSK IS NULL;


-- Q5: Total revenue contribution % by each category
SELECT
    p.Category,
    ROUND(SUM(f.Amount), 2)                                        AS Category_Revenue,
    ROUND(SUM(f.Amount) * 100.0 / SUM(SUM(f.Amount)) OVER (), 2)  AS Revenue_Pct
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimProduct p ON f.ProductSK = p.ProductSK
GROUP BY p.Category
ORDER BY Category_Revenue DESC;


-- Q6: Customers who purchased more than 3 times
SELECT
    c.CustomerID,
    c.CustomerName,
    c.City,
    COUNT(f.TransactionID)                    AS Total_Orders
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimCustomer c
    ON f.CustomerSK = c.CustomerSK
    AND c.IsActive  = 1
GROUP BY c.CustomerID, c.CustomerName, c.City
HAVING Total_Orders > 3
ORDER BY Total_Orders DESC;


-- Q7: Week-over-week sales trend
SELECT
    WEEKOFYEAR(TxnDate)                       AS Week_Number,
    COUNT(TransactionID)                      AS Total_Transactions,
    ROUND(SUM(Amount), 2)                     AS Weekly_Revenue
FROM retail_sales_catalog.silver.FactSales
GROUP BY WEEKOFYEAR(TxnDate)
ORDER BY Week_Number;